# الدرس 08 - نمط تصميم الوكلاء المتعددين


## الإعداد


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## لماذا أنظمة متعددة العوامل؟

تشمل المهام الواقعية مثل تخطيط الرحلات العديد من أنواع الخبرات المختلفة — اللوجستيات، المعرفة المحلية، الميزانية، وأكثر. يحاول وكيل واحد التعامل مع كل شيء بسرعة يصبح من الصعب التحكم به.

تحل أنظمة متعددة العوامل هذه المشكلة من خلال **التخصص**: يركز كل وكيل على مجال خبرة واحد، مما ينتج نتائج ذات جودة أعلى من المتخصص العام. كما أنها تحسن **القابلية للتوسع** — يمكنك إضافة وكلاء جدد (مثل أخصائي الرحلات الجوية، ناقد المطاعم) دون إعادة كتابة سير العمل الحالي. يتعاون الوكلاء معًا من خلال خط أنابيب منظم، حيث يتم تمرير السياق من واحد إلى الآخر.


## إنشاء وكلاء متخصصين


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## بناء تدفق عمل تسلسلي

يتيح لك `WorkflowBuilder` ربط الوكلاء في رسم بياني موجه. هنا نقوم بإنشاء خط أنابيب بسيط من خطوتين: يقوم **مخطط السفر** بصياغة خط سير الرحلة، ثم يقوم **مساعد السفر** بمراجعتها وتحسينها.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## إضافة المزيد من الوكلاء إلى سير العمل

واحدة من أكبر مزايا نمط الوكلاء المتعددين هي سهولة التوسع فيه. أدناه نضيف وكيل **BudgetReviewer** الذي يتحقق من الخطة مقابل ميزانية المسافر، ويشير إلى العناصر التي قد تتسبب في تجاوز التكاليف للحد المسموح، ويقترح بدائل موفرة للمال. يعمل سير العمل الآن على تشغيل ثلاثة وكلاء بالتسلسل:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## الملخص

في هذا الدرس تعلمت كيفية:

1. **إنشاء وكلاء متخصصين** — كلٌ له دور محدد (التخطيط، الاستقبال، مراجعة الميزانية).
2. **ربط الوكلاء في سير عمل متسلسل** باستخدام `WorkflowBuilder` و `add_edge`.
3. **بث الإخراج** من خط أنابيب متعدد الوكلاء، مع تتبع الوكيل الذي يتحدث.
4. **تمديد سير العمل** بإضافة وكلاء جدد إلى السلسلة دون تعديل الوكلاء الموجودين.

نمط تصميم الوكلاء المتعددين يحافظ على بساطة كل وكيل مع إنتاج نتائج أغنى وأكثر مراجعة بدقة مما يمكن لأي وكيل فردي تحقيقه بمفرده.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**إخلاء المسؤولية**:  
تمت ترجمة هذا المستند باستخدام خدمة الترجمة الآلية [Co-op Translator](https://github.com/Azure/co-op-translator). رغم سعينا للدقة، يرجى مراعاة أن الترجمات الآلية قد تحتوي على أخطاء أو عدم دقة. يجب اعتبار المستند الأصلي بلغته الأصلية المصدر الموثوق والمعتمد. للمعلومات الحساسة أو الهامة، يُنصح باللجوء إلى ترجمة بشرية مهنية. نحن غير مسؤولين عن أي سوء فهم أو تفسير خاطئ ينشأ عن استخدام هذه الترجمة.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
